In [9]:
import os

In [10]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [11]:
os.chdir('../')

In [12]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [13]:
# PReparing Data Ingestion entity and config
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    INGESTION_REPORT: Path

In [14]:
from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [15]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    # Data Ingestion related configuration

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=Path(config.source_URL),
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            INGESTION_REPORT=Path(config.INGESTION_REPORT)
        )
        
        return data_ingestion_config

In [16]:
## Components of Data Ingestion
import os
import urllib.request as request
import zipfile
from wdmproject.utils.common import save_json, load_json, save_bin, load_bin, get_size
from wdmproject import logger
import json
from datetime import datetime

In [27]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        self.ingestion_report = {}
        

    ## Download Data
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"file downloaded successfully and saved at: {filename}\n{headers}")
            logger.info(f"file size: {get_size(Path(filename))}")

            self.ingestion_report['dataset_download'] = {
                 "status" : "Success",
                 "file_name" : filename,
                 "file_size" : get_size(Path(filename))
            }

        else:
            logger.info(f"file already exists of size: {get_size(Path(self.config.local_data_file))}")
            logger.info(f"file already exists at: {self.config.local_data_file}")

            self.ingestion_report['dataset_download'] = {
                "status" : "Warning",
                "mesaage" : "File already exists.",
                "file_size" : get_size(self.config.local_data_file)
            }

    ## Extract Data
    def extract_zip_file(self):
        """
        zip_file_path: str 
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"file extracted successfully at: {self.config.unzip_dir}")

        self.ingestion_report['dataset_extraction'] = {
                 "status" : "Success",
                 "message" : "File Extracted Successfully."
        }



    ## Save Ingestion Report
    def save_ingestion_report(self):
            report_path = self.config.INGESTION_REPORT

            with open(report_path, "w") as f:

                json.dump(self.ingestion_report, f, indent=4)

            logger.info(f"Ingestion report saved at {report_path}")


    ## Inititate Data Ingestion

    def initiate_data_ingestion(self):
        try: 
            
            ## Intiating Data Ingestion stage

            self.ingestion_report["stage_metadata"] = {
                "stage_name" : "Data Ingestion",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Data Ingestion Stage.")

            ## Saving start_time
            start_time = datetime.now()

            ## Downloading Dataset
            self.download_file()

            ## Extracting Data
            self.extract_zip_file()

            ## Stage Success
            self.ingestion_report["stage_metadata"]["stage_status"] = "Success"
            self.ingestion_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
             
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.ingestion_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Ingestion Report JSON
            self.save_ingestion_report()

            logger.info("Data Ingestion Checks Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Data Ingestion Error : {e}")

            end_time = datetime.now()

            self.ingestion_report["stage_metadata"]["stage_status"] = "Failed"
            self.ingestion_report["stage_metadata"]["end_time"] = str(end_time)
            self.ingestion_report["stage_metadata"]["error"] = str(e)

            self.save_ingestion_report()

            raise e


In [28]:
# Create a pipeline of data ingestion
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.initiate_data_ingestion()
    logger.info(f"Data Ingestion Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Data Ingestion Pipeline Failed: {e}")
    logger.exception(e)

[2026-03-10 17:14:44,588: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-10 17:14:44,592: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-10 17:14:44,598: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-10 17:14:44,610: INFO: common: created directory at: artifacts]
[2026-03-10 17:14:44,613: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-10 17:14:44,614: INFO: 3830864033: Initiating Data Ingestion Stage.]
[2026-03-10 17:14:44,617: INFO: 3830864033: file already exists of size: ~ 354 KB]
[2026-03-10 17:14:44,619: INFO: 3830864033: file already exists at: artifacts\data_ingestion\data.zip]
[2026-03-10 17:14:44,630: INFO: 3830864033: file extracted successfully at: artifacts\data_ingestion]
[2026-03-10 17:14:44,636: INFO: 3830864033: Ingestion report saved at artifacts\data_ingestion\ingestion_report.json]
[2026-03-10 17:14:44,639: INFO: 3830864033: Data Ingestion Checks Completed Successfully.]
[2026